## LSTM para previsão de preço de café

# 1. Preparação do ambiente

In [13]:
from IPython.display import display, HTML

display(HTML("<style>.container {width: 100% !important;}</style>"))

## 1.1. Importação de bibliotecas

In [14]:
import yfinance as yf
import torch

import matplotlib.pyplot as plt
import pandas as pd

## 1.2. Checagem de GPU

In [15]:
if torch.cuda.is_available():
    print("__CUDNN VERSION:", torch.backends.cudnn.version())
    print("Device Name:", torch.cuda.get_device_name(0))
    device = 'cuda'
else:
    print("CUDA is not available.")
    device = 'cpu'

print('Device:', device)

__CUDNN VERSION: 91900
Device Name: NVIDIA GeForce RTX 5070
Device: cuda


# 2. Base de dados

## 2.1. Baixar preços de café e dolar

In [16]:
raw = yf.download(
    ["KC=F", "BRL=X"],
    start="2019-01-01",
    end="2025-12-31",
    interval="1d"
)

raw

[*********************100%***********************]  2 of 2 completed


Price        Close                High                   Low              \
Ticker       BRL=X        KC=F   BRL=X        KC=F     BRL=X        KC=F   
Date                                                                       
2019-01-01  3.8800         NaN  3.8800         NaN  3.879900         NaN   
2019-01-02  3.8799   99.500000  3.8959  102.650002  3.804300   99.349998   
2019-01-03  3.7863  102.150002  3.8047  103.250000  3.737700   99.449997   
2019-01-04  3.7551  101.599998  3.7836  103.000000  3.711500  100.400002   
2019-01-07  3.6612  102.750000  3.7229  103.449997  3.690600  101.150002   
...            ...         ...     ...         ...       ...         ...   
2025-12-23  5.5900  346.950012  5.6168  352.399994  5.528969  345.600006   
2025-12-24  5.5185  345.149994  5.5227  348.649994  5.512667  343.049988   
2025-12-26  5.5195  350.250000  5.5650  350.700012  5.514800  344.299988   
2025-12-29  5.5425  352.149994  5.5841  354.299988  5.535520  347.049988   
2025-12-30  5.5691  350.200012  5.5777  359.299988  5.481326  349.500000   

Price         Open             Volume           
Ticker       BRL=X        KC=F  BRL=X     KC=F  
Date                                            
2019-01-01  3.8800         NaN    0.0      NaN  
2019-01-02  3.8799  101.599998    0.0  22581.0  
2019-01-03  3.7866   99.500000    0.0  22234.0  
2019-01-04  3.7550  102.300003    0.0  16759.0  
2019-01-07  3.7137  101.500000    0.0  16121.0  
...            ...         ...    ...      ...  
2025-12-23  5.5900  350.000000    0.0  11078.0  
2025-12-24  5.5205  348.100006    0.0      0.0  
2025-12-26  5.5148  345.799988    0.0   7348.0  
2025-12-29  5.5428  349.500000    0.0   9184.0  
2025-12-30  5.5721  350.549988    0.0  11564.0  

[1823 rows x 10 columns]

In [17]:
df_market = raw["Close"].copy()
df_market.columns.name = None
df_market

,BRL=X,KC=F
Date,,
2019-01-01,3.8800,NaN
2019-01-02,3.8799,99.500000
2019-01-03,3.7863,102.150002
2019-01-04,3.7551,101.599998
2019-01-07,3.6612,102.750000
...,...,...
2025-12-23,5.5900,346.950012
2025-12-24,5.5185,345.149994
2025-12-26,5.5195,350.250000


In [18]:
df_market = df_market.rename(columns={
    "KC=F":  "preco_cafe",
    "BRL=X": "cotacao_dolar",
})

df_market

,cotacao_dolar,preco_cafe
Date,,
2019-01-01,3.8800,NaN
2019-01-02,3.8799,99.500000
2019-01-03,3.7863,102.150002
2019-01-04,3.7551,101.599998
2019-01-07,3.6612,102.750000
...,...,...
2025-12-23,5.5900,346.950012
2025-12-24,5.5185,345.149994
2025-12-26,5.5195,350.250000


In [20]:
df_market.index = pd.to_datetime(df_market.index).tz_localize(None)
df_market.index.name = "date"
df_market = df_market[["preco_cafe", "cotacao_dolar"]].dropna(how="all")

df_market.to_csv("data/market.csv", index=False)
df_market

,preco_cafe,cotacao_dolar
date,,
2019-01-01,NaN,3.8800
2019-01-02,99.500000,3.8799
2019-01-03,102.150002,3.7863
2019-01-04,101.599998,3.7551
2019-01-07,102.750000,3.6612
...,...,...
2025-12-23,346.950012,5.5900
2025-12-24,345.149994,5.5185
2025-12-26,350.250000,5.5195


## 2.2. Explorar base de dados

In [ ]:
plt.plot(df_market['preco_cafe'])
plt.title('Coffee')
plt.xlabel("Data")
plt.ylabel('preço')
plt.show()

In [ ]:
plt.plot(df_market['cotacao_dolar'])
plt.title('Dollar')
plt.xlabel("Data")
plt.ylabel('preço')
plt.show()

In [ ]:
df_market.hist()

## 2.3. Checagem de valores nulos

In [ ]:
missing_counts = df_market.isnull().sum()
print("Total de dados faltantes por atributo:")
print(missing_counts)

## 2.4. Juntar dataset de clima e café

In [ ]:
df_weather = pd.read_csv('data/climate_imputed.csv', index_col=0)
df_weather = df_weather.reset_index()
df_weather = df_weather.rename(columns={"datetime": "date"})
df_weather["date"] = pd.to_datetime(df_weather["date"])

df_market_reset = df_market.reset_index()

df_weather

In [ ]:
df = df_market.reset_index().merge(
    df_weather.reset_index(),
    on="date",
    how="left"
)
df.dropna(subset=["date", 'preco_cafe', 'cotacao_dolar'], inplace=True)
df.drop('index', axis=1, inplace=True)
df.to_csv('data/coffee.csv', index=False)

df